<a href="https://colab.research.google.com/github/shreyav-23/Redrob-AI-Candidate-Ranker/blob/main/ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Execution Steps

1. Run the dependency installation cell.

2. Upload the required input files to the Colab runtime:

   * `candidates.jsonl`
   * `job_description.docx`

3. Run all notebook cells from top to bottom.

4. The final submission file will be generated as:
   `P_for_Positive.csv`


In [ ]:
# ============================================
# 1. Install Dependencies
# ============================================

!pip -q install sentence-transformers
!pip -q install python-docx

In [ ]:
# ============================================
# 2. Import Libraries
# ============================================

import json
import re
import heapq

import numpy as np
import pandas as pd

from tqdm import tqdm
from docx import Document

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

In [ ]:
import random
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================
# 3. Load Job Description
# ============================================

def load_job_description(path):

    doc = Document(path)

    paragraphs = []

    for paragraph in doc.paragraphs:

        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)


job_description = load_job_description("job_description.docx")

print("Job Description Loaded Successfully!")

In [ ]:
# ============================================
# 4. Candidate Generator
# ============================================

def candidate_generator(path):


    with open(path, "r", encoding="utf-8") as file:

        for line in file:

            if line.strip():

                yield json.loads(line)

In [ ]:
# ============================================
# 5. Text Processing
# ============================================

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#./ ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def candidate_to_text(candidate):

    profile = candidate.get("profile", {})

    parts = [
        profile.get("current_title", ""),
        profile.get("headline", ""),
        profile.get("summary", "")
    ]

    for job in candidate.get("career_history", []):

        parts.append(job.get("title", ""))
        parts.append(job.get("description", "")[:200])

    parts.extend(
        skill.get("name", "")
        for skill in candidate.get("skills", [])
    )

    return "\n".join(parts)


print("Text Processing Functions Ready!")

In [ ]:
# ============================================
# 6. Parse Job Description
# ============================================

TECH_KEYWORDS = [

    "python",
    "transformers",
    "llm",
    "rag",
    "langchain",
    "embeddings",
    "faiss",
    "pinecone",
    "milvus",
    "qdrant",
    "weaviate",
    "elasticsearch",
    "opensearch",
    "lora",
    "qlora",
    "peft",
    "nlp",
    "machine learning",
    "deep learning",
    "sentence transformers",
    "vector database",
    "retrieval",
    "ranking",
    "recommendation"

]

SOFT_SKILLS = [

    "startup",
    "mentor",
    "product",
    "communication",
    "leadership",
    "collaboration"

]

jd_text = clean_text(job_description)

jd_skills = [
    skill
    for skill in TECH_KEYWORDS
    if skill in jd_text
]

jd_soft_skills = [
    skill
    for skill in SOFT_SKILLS
    if skill in jd_text
]

JD_PROFILE = {

    "skills": jd_skills,
    "soft_skills": jd_soft_skills

}

print("JD Parsed Successfully!")

In [ ]:
# ============================================
# 7. Feature Engineering
# ============================================

JD_SKILL_WEIGHTS = {
    "embeddings": 10,
    "retrieval": 10,
    "ranking": 10,
    "vector database": 10,
    "faiss": 9,
    "pinecone": 9,
    "milvus": 9,
    "qdrant": 9,
    "weaviate": 9,
    "elasticsearch": 8,
    "opensearch": 8,
    "llm": 8,
    "rag": 8,
    "sentence transformers": 8,
    "langchain": 7,
    "transformers": 7,
    "lora": 6,
    "qlora": 6,
    "peft": 6,
    "python": 5,
    "nlp": 5,
    "recommendation": 5,
    "machine learning":7,
"semantic search":9,
"candidate matching":7,
"search":6,
"model serving":7,
"mlops":6,
"kubernetes":5,
"docker":5,
"spark":4,
"airflow":4
}

# ============================================
# Skill Aliases
# ============================================

SKILL_ALIASES = {

    "embeddings": [
        "embedding",
        "embeddings",
        "sentence transformers",
        "sentence-transformers",
        "openai embeddings",
        "bge",
        "e5"
    ],

    "retrieval": [
        "retrieval",
        "information retrieval",
        "semantic search",
        "vector search"
    ],

    "ranking": [
        "ranking",
        "learning to rank",
        "learning-to-rank",
        "ltr"
    ],

    "llm": [
        "llm",
        "large language model",
        "large language models",
        "fine tuning llms",
        "fine-tuning llms"
    ],

    "rag": [
        "rag",
        "retrieval augmented generation",
        "retrieval-augmented generation"
    ],

    "vector database": [
        "vector database",
        "vector databases",
        "vector db",
        "pgvector"
    ],

    "recommendation": [
        "recommendation",
        "recommendation systems",
        "recommender systems",
        "recsys"
    ],

    "nlp": [
        "nlp",
        "natural language processing"
    ],
    "machine learning":[
      "machine learning",
      "ml",
      "artificial intelligence",
      "ai"
    ],

    "semantic search":[
      "semantic search",
      "vector search",
      "dense retrieval"
    ],

    "candidate matching":[
      "candidate matching",
      "talent matching",
      "talent search"
    ],

    "search":[
      "search",
      "information retrieval",
      "retrieval"
    ]
}


def jd_skill_score(candidate):

    career_text = candidate.get("_career_text")
    skills_text = candidate.get("_skills_text")

    if career_text is None:

        career_text = " ".join(
            (
                job.get("title", "") + " " +
                job.get("description", "")
            ).lower()
            for job in candidate.get("career_history", [])
        )

    if skills_text is None:

        skills_text = " ".join(
            skill.get("name", "").lower()
            for skill in candidate.get("skills", [])
        )

    score = 0
    matched = []

    for skill, weight in JD_SKILL_WEIGHTS.items():

        aliases = SKILL_ALIASES.get(skill, [skill])

        found = False

        for alias in aliases:

            if alias in career_text:

                score += weight
                matched.append(skill)
                found = True
                break

        if found:
            continue

        for alias in aliases:

            if alias in skills_text:

                score += weight * 0.5
                matched.append(skill)
                break

    return score, matched


def experience_score(years):

    if 5 <= years <= 9:
        return 10

    elif 4 <= years < 5:
        return 8

    elif 3 <= years < 4:
        return 6

    elif 2 <= years < 3:
        return 3

    elif years < 2:
        return 1

    elif 9 < years <= 15:
        return 9

    else:
        return 7


def behavior_score(signals):

    score = 0

    if signals.get("open_to_work_flag", False):
        score += 5

    score += signals.get("recruiter_response_rate", 0) * 6
    score += signals.get("interview_completion_rate", 0) * 5

    response_time = signals.get(
        "avg_response_time_hours",
        999
    )

    if response_time <= 24:
        score += 2
    elif response_time <= 72:
        score += 1

    github = signals.get(
        "github_activity_score",
        -1
    )

    if github >= 0:
        score += github / 20

    assessments = signals.get(
        "skill_assessment_scores",
        {}
    )

    if assessments:
        score += (
            sum(assessments.values())
            / len(assessments)
        ) / 50

    score += min(
        signals.get(
            "saved_by_recruiters_30d",
            0
        ) / 5,
        3
    )

    score += min(
        signals.get(
            "search_appearance_30d",
            0
        ) / 50,
        2
    )

    score += (
        signals.get(
            "profile_completeness_score",
            0
        ) / 100
    )

    notice = signals.get(
        "notice_period_days",
        0
    )

    if notice > 30:
        score -= min(
            (notice - 30) / 30,
            4
        )

    return score


from datetime import datetime

REFERENCE_DATE = datetime(2026, 6, 1)


def activity_score(signals):

    last_active = signals.get("last_active_date")

    if not last_active:
        return 0

    try:
        last_active = datetime.strptime(
            last_active,
            "%Y-%m-%d"
        )
    except:
        return 0

    days = (
        REFERENCE_DATE - last_active
    ).days

    if days <= 30:
        return 3
    elif days <= 90:
        return 2
    elif days <= 180:
        return 0

    return -5


PRODUCTION_WORDS = {
    "production","pipeline","deployment","serving","monitoring",
    "evaluation","latency","distributed","scalable","inference",
    "semantic search","vector search","information retrieval",
    "query expansion","nearest neighbor","nearest neighbors",
    "ab testing","drift","retraining","observability",
    "microservices","docker","kubernetes","airflow",
    "spark","ray","mlflow","kubeflow"
}

def production_score(candidate):

    text = candidate.get("_cached_text")

    if text is None:
        text = candidate_to_text(candidate).lower()

    score = sum(
        keyword in text
        for keyword in PRODUCTION_WORDS
    )

    years = relevant_ai_experience(candidate)

    if years >= 8:
        score += 3
    elif years >= 5:
        score += 2

    return score

AI_TITLE_KEYWORDS = {
    "ai engineer":8,
    "machine learning engineer":8,
    "ml engineer":8,
    "applied scientist":8,
    "research scientist":7,
    "data scientist":6,
    "nlp engineer":7,
    "search engineer":7,
    "retrieval engineer":7,
    "ranking engineer":7,
    "recommendation engineer":7,
    "ml architect":8,
    "principal":4,
    "staff":3,
    "lead":3,
    "senior":2
}

def title_score(candidate):

    profile = candidate.get("profile", {})
    title = profile.get("current_title", "").lower()
    years = profile.get("years_of_experience", 0)

    score = 0

    if "machine learning engineer" in title:
        score += 8

    if "ai engineer" in title:
        score += 8

    if "senior" in title:
        score += 2

        if years < 4:
            score -= 2

    if "lead" in title:
        score += 3

        if years < 5:
            score -= 3

    if "principal" in title:
        score += 4

        if years < 8:
            score -= 4

    return max(score, 0)

AI_ROLE_KEYWORDS = {
    "ai",
    "artificial intelligence",
    "machine learning",
    "ml",
    "nlp",
    "llm",
    "rag",
    "retrieval",
    "ranking",
    "search",
    "recommendation",
    "recommender",
    "applied scientist",
    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "data scientist"
}


def relevant_ai_experience(candidate):

    total_months = 0

    for job in candidate.get(
        "career_history",
        []
    ):

        text = (
            job.get("title", "") + " " +
            job.get("description", "")
        ).lower()

        if any(
            keyword in text
            for keyword in AI_ROLE_KEYWORDS
        ):
            total_months += job.get(
                "duration_months",
                0
            )

    if total_months == 0:
        return candidate.get(
            "profile",
            {}
        ).get(
            "years_of_experience",
            0
        )

    return total_months / 12


CONSULTING_COMPANIES = {
    "tcs",
    "tata consultancy services",
    "infosys",
    "wipro",
    "cognizant",
    "capgemini",
    "accenture",
    "tech mahindra",
    "hcl",
    "hcl technologies",
    "ibm consulting",
    "ltimindtree",
    "mindtree",
    "l&t technology services",
    "ltts"
}


def consulting_penalty(candidate):

    jobs = candidate.get(
        "career_history",
        []
    )

    if not jobs:
        return 0

    consulting_jobs = 0

    for job in jobs:

        company = job.get(
            "company",
            ""
        ).lower()

        if any(
            firm in company
            for firm in CONSULTING_COMPANIES
        ):
            consulting_jobs += 1

    if consulting_jobs == len(jobs):
        return -15

    return 0

# ============================================
# Profile Consistency Penalty
# ============================================

from datetime import datetime

REFERENCE_DATE = datetime(2026, 6, 1)

SENIOR_TITLES = {
    "senior",
    "staff",
    "lead",
    "principal",
    "architect",
    "manager",
    "director",
    "head",
    "vp"
}

def profile_consistency_penalty(candidate):

    penalty = 0

    profile = candidate.get("profile", {})
    jobs = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    signals = candidate.get("redrob_signals", {})

    claimed_years = profile.get("years_of_experience", 0)

    # =====================================================
    # 1. Experience vs Career History
    # =====================================================

    total_months = sum(
        max(0, job.get("duration_months", 0))
        for job in jobs
    )

    career_years = total_months / 12

    allowed_gap = max(2, claimed_years * 0.30)

    if abs(claimed_years - career_years) > allowed_gap:
        penalty -= 5

    # =====================================================
    # 2. Employment Date Validation
    # =====================================================

    parsed_jobs = []

    for job in jobs:

        start = job.get("start_date")
        end = job.get("end_date")
        is_current = job.get("is_current", False)

        try:
            start_dt = datetime.strptime(start, "%Y-%m-%d")
        except:
            continue

        if is_current or end is None:
            end_dt = REFERENCE_DATE
        else:
            try:
                end_dt = datetime.strptime(end, "%Y-%m-%d")
            except:
                continue

        if end_dt < start_dt:
            penalty -= 8
            continue

        if start_dt > REFERENCE_DATE:
            penalty -= 6

        if end_dt > REFERENCE_DATE:
            penalty -= 4

        parsed_jobs.append((start_dt, end_dt))

    # =====================================================
    # 3. Overlapping Jobs
    # =====================================================

    parsed_jobs.sort()

    overlaps = 0

    for i in range(1, len(parsed_jobs)):

        prev_end = parsed_jobs[i - 1][1]
        curr_start = parsed_jobs[i][0]

        if curr_start < prev_end:
            overlaps += 1

    penalty -= min(overlaps * 3, 9)

    # =====================================================
    # 4. Duplicate Skills
    # =====================================================

    skill_names = [
        skill.get("name", "").strip().lower()
        for skill in skills
        if skill.get("name")
    ]

    duplicate_skills = len(skill_names) - len(set(skill_names))

    penalty -= min(duplicate_skills, 4)

    # =====================================================
    # 5. Duplicate Work Entries
    # =====================================================

    seen = set()
    duplicates = 0

    for job in jobs:

        key = (
            job.get("company", "").lower(),
            job.get("title", "").lower(),
            job.get("start_date")
        )

        if key in seen:
            duplicates += 1
        else:
            seen.add(key)

    penalty -= min(duplicates * 2, 6)

    # =====================================================
    # 6. Skill Duration vs Experience
    # =====================================================

    max_skill_months = int(claimed_years * 12 + 12)

    inconsistent_skills = 0

    for skill in skills:

        months = skill.get("duration_months")

        if months is None:
            continue

        if months > max_skill_months:
            inconsistent_skills += 1

    penalty -= min(inconsistent_skills, 5)

    # =====================================================
    # 7. Senior Title vs Experience
    # =====================================================

    title = profile.get("current_title", "").lower()

    title = profile.get("current_title", "").lower()

    if "principal" in title and claimed_years < 10:
      penalty -= 8

    elif "staff" in title and claimed_years < 8:
      penalty -= 6

    elif "architect" in title and claimed_years < 7:
      penalty -= 5

    elif "lead" in title and claimed_years < 5:
      penalty -= 4

    elif "senior" in title and claimed_years < 4:
      penalty -= 4

    elif "manager" in title and claimed_years < 4:
      penalty -= 4

    # =====================================================
    # 8. Profile Completeness
    # =====================================================

    completeness = signals.get(
        "profile_completeness_score",
        100
    )

    if completeness < 50:
        penalty -= 4
    elif completeness < 70:
        penalty -= 2

    summary = profile.get("summary", "").strip()

    if len(summary) < 40:
        penalty -= 1

    if len(jobs) == 0:
        penalty -= 4

    if len(skills) < 3:
        penalty -= 2

    if (
        claimed_years >= 5 and
        len(jobs) <= 1
    ):
        penalty -= 3

    return penalty

# ============================================
# Fast Candidate Score
# ============================================

def fast_score(candidate):

    profile = candidate.get("profile", {})

    skill_score, _ = jd_skill_score(candidate)

    relevant_years = relevant_ai_experience(candidate)

    exp_score = experience_score(
        relevant_years
    )

    behaviour = behavior_score(
        candidate.get(
            "redrob_signals",
            {}
        )
    )

    behaviour += activity_score(
        candidate.get(
            "redrob_signals",
            {}
        )
    )

    production = production_score(candidate)

    title = title_score(candidate)

    consulting = consulting_penalty(candidate)

    consistency = profile_consistency_penalty(candidate)

    return (
        skill_score * 7
        + exp_score * 2
        + behaviour
        + production * 4
        + title * 3
        + consulting
        + consistency
    )

In [ ]:
# ============================================
# 8. Candidate Retrieval
# ============================================

TOP_PRESELECT = 500
heap = []

for candidate in tqdm(candidate_generator("candidates.jsonl")):

    candidate["_career_text"] = " ".join(
        (
            f"{job.get('title','')} "
            f"{job.get('description','')} "
            f"{job.get('industry','')}"
        ).lower()
        for job in candidate.get("career_history", [])
    )

    candidate["_skills_text"] = " ".join(
        skill.get("name","").lower()
        for skill in candidate.get("skills", [])
    )

    candidate["_cached_text"] = (
        candidate["_career_text"]
        + " "
        + candidate["_skills_text"]
    )

    score = fast_score(candidate)

    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    if profile.get("summary"):
        score += 1

    if profile.get("headline"):
        score += 0.5

    if len(candidate.get("career_history", [])) >= 2:
        score += 1

    if len(candidate.get("skills", [])) >= 5:
        score += 1

    if len(candidate.get("education", [])) > 0:
        score += 0.5

    if signals.get("verified_email", False):
        score += 0.25

    if signals.get("verified_phone", False):
        score += 0.25

    if signals.get("linkedin_connected", False):
        score += 0.5

    if (
        len(candidate.get("career_history", [])) == 0
        and len(candidate.get("skills", [])) < 3
    ):
        score -= 10

    skill_names = [
        s.get("name","").strip().lower()
        for s in candidate.get("skills", [])
    ]

    duplicate_skills = len(skill_names) - len(set(skill_names))
    score -= min(duplicate_skills, 3)

    score += (hash(candidate["candidate_id"]) % 1000) * 1e-8

    item = (
        score,
        candidate["candidate_id"],
        candidate
    )

    if len(heap) < TOP_PRESELECT:
        heapq.heappush(heap, item)
    else:
        heapq.heappushpop(heap, item)

preselected = sorted(heap, reverse=True)

print(f"Retrieved {len(preselected)} candidates.")
print(f"Fast-score cutoff: {preselected[-1][0]:.2f}")

In [ ]:
# ============================================
# 9. Load Embedding Model
# ============================================

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding Model Loaded!")

In [ ]:
# ============================================
# 10. Semantic Embeddings (Fast)
# ============================================

semantic_jd = """
Senior AI Engineer
Production Machine Learning
Information Retrieval
Semantic Search
Vector Search
Embeddings
Ranking
Recommendation Systems
Python
NLP
LLMs
RAG
Sentence Transformers
FAISS
Pinecone
Milvus
Qdrant
Weaviate
Elasticsearch
OpenSearch
"""

jd_embedding = model.encode(
    semantic_jd,
    normalize_embeddings=True,
    convert_to_numpy=True
)

def candidate_to_semantic_text(candidate):

    profile = candidate["profile"]

    parts = []

    parts.append(profile.get("current_title",""))
    parts.append(profile.get("headline",""))

    summary = profile.get("summary","")
    if summary:
        parts.append(summary[:150])

    parts.append(f"{profile.get('years_of_experience',0)} years experience")

    jobs = candidate.get("career_history",[])[:2]

    for job in jobs:

        parts.append(job.get("title",""))
        parts.append(job.get("industry",""))

        desc = job.get("description","")

        if desc:
            parts.append(desc[:80])

    skills = [
        skill["name"]
        for skill in candidate.get("skills",[])[:15]
    ]

    if skills:
        parts.append(" ".join(skills))

    return " | ".join(parts)

candidate_texts = [
    candidate_to_semantic_text(candidate)
    for _,_,candidate in preselected
]

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=256,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

semantic_scores = np.dot(
    candidate_embeddings,
    jd_embedding
)

print("Semantic Ranking Ready!")

In [ ]:
# ============================================
# 11. Final Ranking
# ============================================

fast_scores = np.array([x[0] for x in preselected], dtype=float)

fast_min = fast_scores.min()
fast_max = fast_scores.max()

fast_range = fast_max - fast_min

final_rankings = []

for semantic, (fast, cid, candidate) in zip(
    semantic_scores,
    preselected
):

    if fast_range > 1e-9:
        fast_norm = (fast - fast_min) / fast_range
    else:
        fast_norm = 1.0

    final_score = (
        0.60 * fast_norm +
        0.40 * float(semantic)
    )

    final_rankings.append(
        (
            final_score,
            cid,
            candidate,
            float(semantic),
            float(fast)
        )
    )

final_rankings.sort(
    key=lambda x: (-x[0], x[1])
)

print(f"Final Ranking Complete! Ranked {len(final_rankings)} candidates.")

In [ ]:
for score, cid, candidate, semantic, fast in final_rankings[25:40]:
    print(
        cid,
        candidate["profile"]["current_title"],
        candidate["profile"]["years_of_experience"],
        f"fast={fast:.2f}",
        f"semantic={semantic:.3f}",
        f"final={score:.3f}"
    )

In [ ]:
# ============================================
# 12. Reason Generation
# ============================================

def candidate_specialization(candidate):

    career = candidate.get("_career_text", "").lower()
    skills = candidate.get("_skills_text", "").lower()
    title = candidate.get("profile", {}).get("current_title", "").lower()

    # -------------------------------
    # Strong title-based specialization
    # -------------------------------

    if any(x in title for x in [
        "search engineer",
        "search scientist",
        "retrieval engineer"
    ]):
        return "search and retrieval"

    if any(x in title for x in [
        "recommendation engineer",
        "recommendation scientist",
        "recommender"
    ]):
        return "recommendation systems"

    if "nlp" in title:
        return "NLP"

    if any(x in title for x in [
        "mlops",
        "platform engineer",
        "infrastructure engineer"
    ]):
        return "production AI systems"

    if any(x in title for x in [
        "genai",
        "llm",
        "generative ai"
    ]):
        return "LLM applications"

    scores = {
        "production AI systems": 0,
        "recommendation systems": 0,
        "search and retrieval": 0,
        "NLP": 0,
        "LLM applications": 0,
        "vector databases": 0
    }

    # -------------------------------
    # Production AI
    # -------------------------------

    production_keywords = [
        "production",
        "deployment",
        "deploy",
        "serving",
        "mlops",
        "pipeline",
        "docker",
        "kubernetes",
        "distributed",
        "real-time",
        "scalable",
        "monitoring",
        "inference",
        "model serving"
    ]

    for word in production_keywords:
        if word in career:
            scores["production AI systems"] += 2

    # -------------------------------
    # Search & Retrieval
    # -------------------------------

    retrieval_keywords = [
        "semantic search",
        "information retrieval",
        "dense retrieval",
        "hybrid retrieval",
        "vector search",
        "retrieval augmented generation",
        "retrieval-augmented generation",
        "bm25",
        "elasticsearch",
        "opensearch"
    ]

    for word in retrieval_keywords:
        if word in career:
            scores["search and retrieval"] += 3

    if (
        "search" in career and
        (
            "retrieval" in career or
            "ranking" in career or
            "elasticsearch" in career or
            "opensearch" in career
        )
    ):
        scores["search and retrieval"] += 2

    # -------------------------------
    # Recommendation Systems
    # -------------------------------

    recommendation_keywords = [
        "recommendation system",
        "recommendation engine",
        "recommender system",
        "recommender engine",
        "personalization"
    ]

    for word in recommendation_keywords:
        if word in career:
            scores["recommendation systems"] += 3

    if (
        "recommendation" in career and
        "model" in career
    ):
        scores["recommendation systems"] += 2

    # -------------------------------
    # NLP
    # -------------------------------

    nlp_keywords = [
        "natural language processing",
        "text classification",
        "named entity recognition",
        "question answering",
        "sentiment analysis",
        "bert",
        "roberta",
        "token classification",
        "tokenization"
    ]

    for word in nlp_keywords:
        if word in career:
            scores["NLP"] += 3

    # -------------------------------
    # LLM Applications
    # -------------------------------

    llm_keywords = [
        "llm",
        "rag",
        "langchain",
        "prompt engineering",
        "prompt",
        "lora",
        "qlora",
        "peft",
        "fine tuning",
        "fine-tuning",
        "instruction tuning",
        "agent",
        "agents"
    ]

    for word in llm_keywords:
        if word in career:
            scores["LLM applications"] += 3

        if word in skills:
            scores["LLM applications"] += 1

    # -------------------------------
    # Vector Databases
    # -------------------------------

    vector_hits = 0

    for tech in [
        "faiss",
        "pinecone",
        "milvus",
        "qdrant",
        "weaviate"
    ]:

        if tech in career:
            vector_hits += 2

        if tech in skills:
            vector_hits += 1

    scores["vector databases"] = vector_hits

    # -------------------------------
    # Vector DB is fallback only
    # -------------------------------

    other_max = max(
        value
        for key, value in scores.items()
        if key != "vector databases"
    )

    if other_max >= scores["vector databases"]:
        scores["vector databases"] = 0

    # -------------------------------
    # Require reasonable confidence
    # -------------------------------

    best = max(scores, key=scores.get)
    best_score = scores[best]

    sorted_scores = sorted(scores.values(), reverse=True)

    if best_score < 4:
      return None

# Require the winner to be clearly ahead
    if len(sorted_scores) > 1 and best_score - sorted_scores[1] < 2:
      return None

    return best


def generate_reason(candidate, semantic_score):
    profile=candidate.get("profile",{})

    signals=candidate.get("redrob_signals",{})

    title=profile.get("current_title","Professional")

    exp=profile.get("years_of_experience",0)

    parts=[f"{title} with {exp:.1f} years of experience"]

    spec=candidate_specialization(candidate)

    if spec:
        parts[-1]+=f" in {spec}"

    if signals.get("open_to_work_flag",False):
        parts.append("open to work")
    rr=signals.get("recruiter_response_rate",0)

    if rr>=0.90:
        parts.append(f"recruiter response rate {rr:.2f}")
    notice=signals.get("notice_period_days",0)

    if notice>=90:
        parts.append(f"{notice}-day notice period")

    if exp<3:
        parts.append("early-career candidate with relevant AI experience")

    elif 3<=exp<5:
        parts.append("approaching the preferred experience range")

    elif 12<exp<=15:
        parts.append("additional senior-level experience")

    elif exp>15:
        parts.append("experience above the preferred range")

    return "; ".join(parts)+"."

In [ ]:
# ============================================
# 13. Generate Submission
# ============================================

submission_rows = []
seen = set()
last_score = 1.0

for rank, (
    score,
    cid,
    candidate,
    semantic,
    fast
) in enumerate(final_rankings, start=1):

    if rank > 100:
        break

    if cid in seen:
        continue

    seen.add(cid)

    score = max(0.0, min(float(score), 1.0))

    if score > last_score:
        score = last_score

    last_score = score

    submission_rows.append({
        "candidate_id": cid,
        "rank": rank,
        "score": round(score, 6),
        "reasoning": generate_reason(
            candidate,
            semantic
        )
    })

submission = pd.DataFrame(submission_rows)

submission = submission.sort_values(
    "rank"
).reset_index(drop=True)

assert len(submission) == 100
assert submission["candidate_id"].is_unique
assert list(submission["rank"]) == list(range(1,101))

assert (
    submission["score"]
    .diff()
    .fillna(0)
    <= 1e-9
).all()

print(f"Generated {len(submission)} ranked candidates.")

submission.head(10)

In [ ]:
# ============================================
# 14. Save Submission
# ============================================

OUTPUT_FILE = "P_for_Positive.csv"

submission.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Submission saved to {OUTPUT_FILE}")

In [ ]:
# ============================================
# 15. Validate & Download Submission
# ============================================

from google.colab import files

required_columns = [
    "candidate_id",
    "rank",
    "score",
    "reasoning"
]

assert list(submission.columns) == required_columns
assert len(submission) == 100
assert submission["candidate_id"].is_unique
assert submission["candidate_id"].str.match(r"^CAND_\d{7}$").all()
assert list(submission["rank"]) == list(range(1,101))
assert submission["score"].between(0,1).all()

assert (
    submission["score"]
    .diff()
    .fillna(0)
    <= 1e-9
).all()

assert submission["reasoning"].str.len().gt(20).all()

print("✅ Submission validation passed.")

OUTPUT_FILE = "P_for_Positive.csv"
submission.to_csv(OUTPUT_FILE, index=False)

print(f"Saved as {OUTPUT_FILE}")

files.download(OUTPUT_FILE)